# VAR-d20 Paper-Inspired PFB Style Injection

This notebook adapts the methodology from **A Training-Free Style-Personalization via SVD-Based Feature Decomposition** to our image-only VAR-d20 setting.

The paper uses Infinity with text + reference style image. Our current experiment has only:

```text
content image + style image
```

So we cannot copy the paper's text-conditioned Infinity pipeline exactly. Instead, we implement the closest valid version inside the frozen VAR VAE feature pyramid:

```text
content image -> content VAE feature path
style image   -> style VAE feature path
generation    -> content feature path modified by PFB
```

The main experiment is still single-scale: inject style at scale 0 only, scale 1 only, ..., scale 9 only. This lets us find which VAR-d20 scale behaves like the paper's pivotal style-sensitive stage.


## Method Mapping From The Paper To Our VAR Experiment

The paper's method has two key modules:

```text
PFB: Principal Feature Blending
SAC: Structural Attention Correction
```

Our adaptation:

```text
PFB -> implemented directly on VAR VAE accumulated latent features fhat_s
SAC -> approximated by keeping later content residuals after the PFB scale
```

Why SAC is only approximated here:

```text
Infinity has text conditioning, generation branches, and attention Q/K intervention.
VAR-d20 is class-conditioned and our notebook is image-only reconstruction/editing through the VAE.
```

So this notebook tests the part we can implement cleanly without modifying VAR source code: SVD-based PFB feature decomposition and blending.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LeeHoang2710/Style-Transfer-Experiment.git"
BRANCH = "main"

if Path('/kaggle/working').exists():
    RUNTIME_ROOT = Path('/kaggle/working')
elif Path('/content').exists():
    RUNTIME_ROOT = Path('/content')
else:
    RUNTIME_ROOT = Path.cwd()

WORKSPACE = RUNTIME_ROOT / 'VAR_Style_Transfer_Workspace'

if not WORKSPACE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKSPACE), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(WORKSPACE), 'reset', '--hard', f'origin/{BRANCH}'], check=True)

print('Workspace synced to:')
subprocess.run(['git', '-C', str(WORKSPACE), 'log', '--oneline', '-1'], check=True)

os.chdir(WORKSPACE / 'VAR')
print('Current directory:', Path.cwd())


In [ ]:
!nvidia-smi
!pip install -q huggingface_hub einops matplotlib pandas tqdm


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

weights_dir = RUNTIME_ROOT / 'VAR_weights'
weights_dir.mkdir(parents=True, exist_ok=True)

vae_path = hf_hub_download(
    repo_id='FoundationVision/var',
    filename='vae_ch160v4096z32.pth',
    local_dir=weights_dir,
)
var_path = hf_hub_download(
    repo_id='FoundationVision/var',
    filename='var_d20.pth',
    local_dir=weights_dir,
)

print('VAE:', vae_path)
print('VAR:', var_path)


In [ ]:
import gc
import math
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torchvision.transforms.functional as TF
from torchvision.utils import make_grid
from tqdm.auto import tqdm
from models import build_vae_var

MODEL_DEPTH = 20
patch_nums = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

vae, var = build_vae_var(
    V=4096,
    Cvae=32,
    ch=160,
    share_quant_resi=4,
    device=device,
    patch_nums=patch_nums,
    num_classes=1000,
    depth=MODEL_DEPTH,
    shared_aln=False,
)

vae.load_state_dict(torch.load(vae_path, map_location='cpu'), strict=True)
var.load_state_dict(torch.load(var_path, map_location='cpu'), strict=True)
vae.eval(); var.eval()
for p in vae.parameters():
    p.requires_grad_(False)
for p in var.parameters():
    p.requires_grad_(False)

print('device:', device)
print('patch_nums:', patch_nums)


## Data And Image Helpers

Use semantically compatible content/style pairs first. This matters because the paper's style reference is not supposed to replace the content object. If the reference image has totally unrelated geometry, leakage is much harder to interpret.


In [ ]:
DATA_ROOT = WORKSPACE
CONTENT_DIR = DATA_ROOT / 'content'
STYLE_DIR = DATA_ROOT / 'style'
OUT_DIR = RUNTIME_ROOT / 'VAR_outputs' / 'paper_pfb_feature_injection'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('content images:', len(sorted(CONTENT_DIR.glob('*.png'))))
print('style images:', len(sorted(STYLE_DIR.rglob('*.png'))))
print('outputs:', OUT_DIR)


In [ ]:
def load_var_image(path, size=256):
    img = Image.open(path).convert('RGB')
    img = ImageOps.fit(
        img,
        (size, size),
        method=Image.Resampling.LANCZOS,
        centering=(0.5, 0.5),
    )
    x = TF.to_tensor(img).mul(2).sub(1).unsqueeze(0).to(device)
    return x, img


def tensor_to_pil(x):
    x = x.detach().float().cpu()
    if x.ndim == 4:
        x = x[0]
    x = x.clamp(-1, 1).add(1).div(2)
    x = x.permute(1, 2, 0).numpy()
    return Image.fromarray((x * 255).astype('uint8'))


def show_tensor_img(x, title=None):
    plt.imshow(tensor_to_pil(x))
    if title:
        plt.title(title)
    plt.axis('off')


def show_pil_img(img, title=None):
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')


def save_image_grid(tensors, path, nrow=4):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    grid = make_grid(torch.cat(tensors, dim=0).clamp(-1, 1).add(1).div(2), nrow=nrow, padding=2)
    grid_img = Image.fromarray((grid.detach().cpu().permute(1, 2, 0).numpy() * 255).astype('uint8'))
    grid_img.save(path)
    return path


In [ ]:
matched_pairs = [
    ('lake + mountain painting', CONTENT_DIR / '0014.png', STYLE_DIR / 'OilPainting' / 'OP005.png'),
    ('mountain + watercolor landscape', CONTENT_DIR / '0011.png', STYLE_DIR / 'WaterColor' / 'WC001.png'),
    ('house + watercolor house', CONTENT_DIR / '0015.png', STYLE_DIR / 'WaterColor' / 'WC002.png'),
    ('clouds + anime sky', CONTENT_DIR / '0004.png', STYLE_DIR / 'AnimeStyle' / 'AS007.png'),
    ('portrait + painted portrait', CONTENT_DIR / '0013.png', STYLE_DIR / 'VanGogh' / 'VanGogh001.png'),
]

for name, content_path, style_path in matched_pairs:
    assert content_path.exists(), content_path
    assert style_path.exists(), style_path

plt.figure(figsize=(10, 2 * len(matched_pairs)))
for row, (name, content_path, style_path) in enumerate(matched_pairs):
    _, content_pil = load_var_image(content_path)
    _, style_pil = load_var_image(style_path)
    plt.subplot(len(matched_pairs), 2, row * 2 + 1)
    show_pil_img(content_pil, f'{name}: content')
    plt.subplot(len(matched_pairs), 2, row * 2 + 2)
    show_pil_img(style_pil, f'{name}: style')
plt.tight_layout()
plt.show()


## Extract VAR VAE Feature Pyramid

VAR's VAE gives us the feature pyramid needed for a paper-style intervention.

For each image:

```text
image -> encoder -> quant_conv -> f
f -> multi-scale quantizer -> fhat_0, fhat_1, ..., fhat_9
```

`fhat_s` is the accumulated latent after scale `s`. We also compute residuals:

```text
delta_0 = fhat_0
delta_s = fhat_s - fhat_{s-1}
```

The paper edits an internal feature at one key generation stage. Our closest VAE-space equivalent is to edit `fhat_s`, then preserve later content residuals.


In [ ]:
@torch.no_grad()
def image_to_fhat_scales(x):
    f = vae.quant_conv(vae.encoder(x))
    return vae.quantize.f_to_idxBl_or_fhat(f, to_fhat=True)


def fhat_scales_to_deltas(fhat_scales):
    deltas = []
    previous = torch.zeros_like(fhat_scales[0])
    for fhat in fhat_scales:
        deltas.append(fhat - previous)
        previous = fhat
    return deltas


@torch.no_grad()
def image_to_features(x):
    fhat_scales = image_to_fhat_scales(x)
    deltas = fhat_scales_to_deltas(fhat_scales)
    recon = vae.fhat_to_img(fhat_scales[-1])
    return {
        'fhat_scales': fhat_scales,
        'deltas': deltas,
        'recon': recon,
    }


@torch.no_grad()
def decode_fhat(fhat):
    return vae.fhat_to_img(fhat)


## Principal Feature Extractor Phi

The paper defines a style extractor using SVD:

```text
F = U Sigma V^T
Phi(F) = U W Sigma V^T
```

`W` is an exponential weight:

```text
W_i = exp(-alpha * i)
```

Large singular components get strong weight. Later components get weaker weight. The idea is that the dominant principal feature carries style-like information, while the residual keeps more content/structure.


In [ ]:
def phi_principal_feature(F_BCHW, alpha=1.0, rank=None, center=False):
    """Paper-style Phi(F) = U W Sigma V^T on each image feature map."""
    dtype = F_BCHW.dtype
    B, C, H, W = F_BCHW.shape
    outputs = []

    for b in range(B):
        F_NC = F_BCHW[b].detach().float().permute(1, 2, 0).reshape(H * W, C)
        if center:
            mean = F_NC.mean(dim=0, keepdim=True)
            F_work = F_NC - mean
        else:
            mean = None
            F_work = F_NC

        U, S, Vh = torch.linalg.svd(F_work, full_matrices=False)
        r = S.shape[0] if rank is None else min(rank, S.shape[0])
        weights = torch.exp(-alpha * torch.arange(r, device=F_work.device, dtype=F_work.dtype))

        phi_NC = (U[:, :r] * (S[:r] * weights).unsqueeze(0)) @ Vh[:r, :]
        if center and mean is not None:
            # Phi is kept as the principal centered component. The residual branch keeps the original mean.
            pass

        phi_BCHW = phi_NC.reshape(H, W, C).permute(2, 0, 1)
        outputs.append(phi_BCHW)

    return torch.stack(outputs, dim=0).to(dtype=dtype)


def remove_content_subspace_from_style(style_feature, content_feature, clean_rank=8, center=True, strength=1.0):
    """Remove content-like channel subspace from the style feature before Phi(S)."""
    dtype = style_feature.dtype
    B, C, H, W = style_feature.shape
    outputs = []

    for b in range(B):
        S_NC = style_feature[b].detach().float().permute(1, 2, 0).reshape(H * W, C)
        C_NC = content_feature[b].detach().float().permute(1, 2, 0).reshape(H * W, C)

        if center:
            S_mean = S_NC.mean(dim=0, keepdim=True)
            C_mean = C_NC.mean(dim=0, keepdim=True)
            S_work = S_NC - S_mean
            C_work = C_NC - C_mean
        else:
            S_mean = None
            S_work = S_NC
            C_work = C_NC

        _, _, Vh_c = torch.linalg.svd(C_work, full_matrices=False)
        r = min(clean_rank, Vh_c.shape[0])
        basis_c = Vh_c[:r].T.contiguous()  # [C, r]

        projected_content_like = (S_work @ basis_c) @ basis_c.T
        S_clean_work = S_work - strength * projected_content_like
        S_clean_NC = S_clean_work + S_mean if center and S_mean is not None else S_clean_work

        outputs.append(S_clean_NC.reshape(H, W, C).permute(2, 0, 1))

    return torch.stack(outputs, dim=0).to(dtype=dtype)


def pfb_blend(
    content_feature,
    style_feature,
    alpha=1.0,
    rank=None,
    strength=1.0,
    center=False,
    clean_style=False,
    clean_rank=8,
    clean_center=True,
    clean_strength=1.0,
):
    """F_mix = Phi(S) + (C - Phi(C)); optionally use Phi(S_clean)."""
    style_for_phi = style_feature
    if clean_style:
        style_for_phi = remove_content_subspace_from_style(
            style_feature=style_feature,
            content_feature=content_feature,
            clean_rank=clean_rank,
            center=clean_center,
            strength=clean_strength,
        )

    phi_content = phi_principal_feature(content_feature, alpha=alpha, rank=rank, center=center)
    phi_style = phi_principal_feature(style_for_phi, alpha=alpha, rank=rank, center=center)
    paper_full = phi_style + (content_feature - phi_content)
    return content_feature + strength * (paper_full - content_feature)


def full_feature_replace(content_fhat_scales, style_fhat_scales, content_deltas, scale_id, keep_later_content=True):
    """A leakage baseline: replace the whole accumulated feature at scale s."""
    mixed_fhat = style_fhat_scales[scale_id].clone()
    if keep_later_content and scale_id + 1 < len(content_deltas):
        mixed_fhat = mixed_fhat + torch.stack(content_deltas[scale_id + 1:], dim=0).sum(dim=0)
    return decode_fhat(mixed_fhat)


def pfb_inject_at_scale(
    content_fhat_scales,
    style_fhat_scales,
    content_deltas,
    scale_id,
    alpha=1.0,
    rank=None,
    strength=1.0,
    center=False,
    keep_later_content=True,
    clean_style=False,
    clean_rank=8,
    clean_center=True,
    clean_strength=1.0,
):
    """Apply PFB at one accumulated scale, then preserve later content residuals."""
    mixed_fhat = pfb_blend(
        content_fhat_scales[scale_id],
        style_fhat_scales[scale_id],
        alpha=alpha,
        rank=rank,
        strength=strength,
        center=center,
        clean_style=clean_style,
        clean_rank=clean_rank,
        clean_center=clean_center,
        clean_strength=clean_strength,
    )

    if keep_later_content and scale_id + 1 < len(content_deltas):
        mixed_fhat = mixed_fhat + torch.stack(content_deltas[scale_id + 1:], dim=0).sum(dim=0)

    return decode_fhat(mixed_fhat)


def pfb_inject_residual_scales(
    content_deltas,
    style_deltas,
    scale_ids,
    alpha=1.0,
    rank=None,
    strength=1.0,
    center=False,
    clean_style=False,
    clean_rank=8,
    clean_center=True,
    clean_strength=1.0,
):
    """Apply PFB to selected residual scales. This avoids double-counting accumulated fhat states."""
    scale_ids = set(scale_ids)
    mixed_deltas = [delta.clone() for delta in content_deltas]

    for scale_id in sorted(scale_ids):
        mixed_deltas[scale_id] = pfb_blend(
            content_deltas[scale_id],
            style_deltas[scale_id],
            alpha=alpha,
            rank=rank,
            strength=strength,
            center=center,
            clean_style=clean_style,
            clean_rank=clean_rank,
            clean_center=clean_center,
            clean_strength=clean_strength,
        )

    mixed_fhat = torch.stack(mixed_deltas, dim=0).sum(dim=0)
    return decode_fhat(mixed_fhat)


## Sanity Check: PFB Versus Full Feature Replacement

This checks the core paper intuition:

```text
Full replacement should transfer style strongly, but risks content leakage.
PFB should transfer principal style while preserving more content residual.
```

If PFB looks almost identical to content, increase `PFB_STRENGTH` or decrease `PFB_ALPHA`. If PFB leaks too much style image structure, increase `PFB_ALPHA` or use a smaller `PFB_RANK`.


In [ ]:
PAIR_ID = 0
PIVOT_SCALE = 3
PFB_ALPHA = 1.0
PFB_RANK = None          # None means use all singular directions with exponential decay
PFB_STRENGTH = 1.0
CENTER_FEATURES = False
KEEP_LATER_CONTENT = True

pair_name, content_path, style_path = matched_pairs[PAIR_ID]
content_x, _ = load_var_image(content_path)
style_x, _ = load_var_image(style_path)

with torch.no_grad():
    content_pack = image_to_features(content_x)
    style_pack = image_to_features(style_x)

    full_replace_img = full_feature_replace(
        content_pack['fhat_scales'],
        style_pack['fhat_scales'],
        content_pack['deltas'],
        scale_id=PIVOT_SCALE,
        keep_later_content=KEEP_LATER_CONTENT,
    )
    pfb_img = pfb_inject_at_scale(
        content_pack['fhat_scales'],
        style_pack['fhat_scales'],
        content_pack['deltas'],
        scale_id=PIVOT_SCALE,
        alpha=PFB_ALPHA,
        rank=PFB_RANK,
        strength=PFB_STRENGTH,
        center=CENTER_FEATURES,
        keep_later_content=KEEP_LATER_CONTENT,
    )

items = [
    ('content recon', content_pack['recon']),
    ('style recon', style_pack['recon']),
    (f'full replace fhat_{PIVOT_SCALE}', full_replace_img),
    (f'PFB fhat_{PIVOT_SCALE}', pfb_img),
]

plt.figure(figsize=(16, 4))
for i, (title, img) in enumerate(items):
    plt.subplot(1, 4, i + 1)
    show_tensor_img(img, title)
plt.suptitle(f'{pair_name} | alpha={PFB_ALPHA}, rank={PFB_RANK}, strength={PFB_STRENGTH}')
plt.tight_layout()
plt.show()

save_path = OUT_DIR / f'sanity_pfb_vs_full_replace_pair{PAIR_ID}_scale{PIVOT_SCALE}.png'
save_image_grid([img for _, img in items], save_path, nrow=4)
print('saved:', save_path)


## Main Test: Single-Scale PFB Sweep

The paper found a pivotal stage in Infinity. We should not assume VAR-d20 has the same stage.

This cell applies PFB at exactly one scale each time:

```text
PFB at scale 0 only
PFB at scale 1 only
...
PFB at scale 9 only
```

Later content residuals are kept after the intervention scale. This is our VAE-space approximation of using the content path to preserve structure after style injection.


In [ ]:
PAIR_ID = 0
PFB_ALPHA = 1.0
PFB_RANK = None
PFB_STRENGTH = 1.0
CENTER_FEATURES = False
KEEP_LATER_CONTENT = True

pair_name, content_path, style_path = matched_pairs[PAIR_ID]
content_x, _ = load_var_image(content_path)
style_x, _ = load_var_image(style_path)

with torch.no_grad():
    content_pack = image_to_features(content_x)
    style_pack = image_to_features(style_x)
    pfb_outputs = [
        pfb_inject_at_scale(
            content_pack['fhat_scales'],
            style_pack['fhat_scales'],
            content_pack['deltas'],
            scale_id=scale_id,
            alpha=PFB_ALPHA,
            rank=PFB_RANK,
            strength=PFB_STRENGTH,
            center=CENTER_FEATURES,
            keep_later_content=KEEP_LATER_CONTENT,
        )
        for scale_id in range(len(patch_nums))
    ]

items = [('content recon', content_pack['recon']), ('style recon', style_pack['recon'])]
items += [(f'PFB scale {i} ({pn}x{pn})', img) for i, (pn, img) in enumerate(zip(patch_nums, pfb_outputs))]
cols = 4
rows = math.ceil(len(items) / cols)

plt.figure(figsize=(4 * cols, 4 * rows))
for i, (title, img) in enumerate(items):
    plt.subplot(rows, cols, i + 1)
    show_tensor_img(img, title)
plt.suptitle(f'{pair_name} | single-scale PFB sweep')
plt.tight_layout()
plt.show()

save_path = OUT_DIR / f'single_pair_pfb_sweep_pair{PAIR_ID}_alpha{PFB_ALPHA}_strength{PFB_STRENGTH}.png'
save_image_grid([img for _, img in items], save_path, nrow=cols)
print('saved:', save_path)


## Single-Scale Route Comparison: Phi(S) Versus Phi(S_clean)

This section keeps the same single-scale sweep, but compares two routes:

```text
normal route:  Phi(S)
clean route:   S_clean = S - projection_of_S_onto_content_subspace, then Phi(S_clean)
```

The clean route tests the CSD-VAR idea that the style reference contains content leakage. If the clean route works, it should reduce style-image structure leakage while keeping useful color/texture style.


In [ ]:
PAIR_ID = 0
PFB_ALPHA = 1.0
PFB_RANK = None
PFB_STRENGTH = 1.0
CLEAN_RANK = 8
CLEAN_STRENGTH = 1.0
CENTER_FEATURES = False
KEEP_LATER_CONTENT = True

pair_name, content_path, style_path = matched_pairs[PAIR_ID]
content_x, _ = load_var_image(content_path)
style_x, _ = load_var_image(style_path)

with torch.no_grad():
    content_pack = image_to_features(content_x)
    style_pack = image_to_features(style_x)

    normal_outputs = []
    clean_outputs = []
    for scale_id in range(len(patch_nums)):
        normal_outputs.append(
            pfb_inject_at_scale(
                content_pack['fhat_scales'],
                style_pack['fhat_scales'],
                content_pack['deltas'],
                scale_id=scale_id,
                alpha=PFB_ALPHA,
                rank=PFB_RANK,
                strength=PFB_STRENGTH,
                center=CENTER_FEATURES,
                keep_later_content=KEEP_LATER_CONTENT,
                clean_style=False,
            )
        )
        clean_outputs.append(
            pfb_inject_at_scale(
                content_pack['fhat_scales'],
                style_pack['fhat_scales'],
                content_pack['deltas'],
                scale_id=scale_id,
                alpha=PFB_ALPHA,
                rank=PFB_RANK,
                strength=PFB_STRENGTH,
                center=CENTER_FEATURES,
                keep_later_content=KEEP_LATER_CONTENT,
                clean_style=True,
                clean_rank=CLEAN_RANK,
                clean_strength=CLEAN_STRENGTH,
            )
        )

cols = len(patch_nums) + 1
plt.figure(figsize=(2.7 * cols, 7.5))

plt.subplot(3, cols, 1)
show_tensor_img(content_pack['recon'], 'content recon')
plt.subplot(3, cols, cols + 1)
show_tensor_img(style_pack['recon'], 'style recon')
plt.subplot(3, cols, 2 * cols + 1)
show_tensor_img(content_pack['recon'], 'content recon')

for scale_id, pn in enumerate(patch_nums):
    plt.subplot(3, cols, scale_id + 2)
    show_tensor_img(normal_outputs[scale_id], f'Phi(S) s{scale_id}\n{pn}x{pn}')
    plt.subplot(3, cols, cols + scale_id + 2)
    show_tensor_img(clean_outputs[scale_id], f'Phi(S_clean) s{scale_id}\n{pn}x{pn}')
    diff = (clean_outputs[scale_id] - normal_outputs[scale_id]).abs().mean(dim=1, keepdim=True)
    diff = diff / diff.amax().clamp_min(1e-6) * 2 - 1
    plt.subplot(3, cols, 2 * cols + scale_id + 2)
    show_tensor_img(diff.repeat(1, 3, 1, 1), f'abs diff s{scale_id}')

plt.suptitle(f'{pair_name} | normal Phi(S) vs cleaned Phi(S_clean)')
plt.tight_layout()
plt.show()

save_path = OUT_DIR / f'route_compare_phi_vs_clean_pair{PAIR_ID}.png'
save_image_grid(
    [content_pack['recon']] + normal_outputs + [style_pack['recon']] + clean_outputs,
    save_path,
    nrow=cols,
)
print('saved:', save_path)


## CSD-VAR Style-Scale Group Injection: 0, 1, 2, 9

CSD-VAR suggests style-sensitive scales include early scales plus the final fine scale.

For VAR-d20, we test the analogous group:

```text
style group = [0, 1, 2, 9]
```

Because group injection touches multiple scales, this section applies PFB to residual features instead of accumulated `fhat_s` features. That avoids counting the same accumulated content/style information multiple times.


In [ ]:
PAIR_ID = 0
PFB_ALPHA = 1.0
PFB_RANK = None
PFB_STRENGTH = 1.0
CLEAN_RANK = 8
CLEAN_STRENGTH = 1.0
CENTER_FEATURES = False

scale_groups_to_test = [
    ('single best placeholder: scale 3', [3]),
    ('CSD style group: 0,1,2,9', [0, 1, 2, 9]),
    ('fine group: 7,8,9', [7, 8, 9]),
    ('coarse group: 0,1,2', [0, 1, 2]),
]

pair_name, content_path, style_path = matched_pairs[PAIR_ID]
content_x, _ = load_var_image(content_path)
style_x, _ = load_var_image(style_path)

with torch.no_grad():
    content_pack = image_to_features(content_x)
    style_pack = image_to_features(style_x)

    items = [('content recon', content_pack['recon']), ('style recon', style_pack['recon'])]
    for group_name, scale_ids in scale_groups_to_test:
        normal_img = pfb_inject_residual_scales(
            content_pack['deltas'],
            style_pack['deltas'],
            scale_ids=scale_ids,
            alpha=PFB_ALPHA,
            rank=PFB_RANK,
            strength=PFB_STRENGTH,
            center=CENTER_FEATURES,
            clean_style=False,
        )
        clean_img = pfb_inject_residual_scales(
            content_pack['deltas'],
            style_pack['deltas'],
            scale_ids=scale_ids,
            alpha=PFB_ALPHA,
            rank=PFB_RANK,
            strength=PFB_STRENGTH,
            center=CENTER_FEATURES,
            clean_style=True,
            clean_rank=CLEAN_RANK,
            clean_strength=CLEAN_STRENGTH,
        )
        items.append((f'{group_name}\nPhi(S)', normal_img))
        items.append((f'{group_name}\nPhi(S_clean)', clean_img))

cols = 5
rows = math.ceil(len(items) / cols)
plt.figure(figsize=(4 * cols, 4 * rows))
for i, (title, img) in enumerate(items):
    plt.subplot(rows, cols, i + 1)
    show_tensor_img(img, title)
plt.suptitle(f'{pair_name} | group PFB injection')
plt.tight_layout()
plt.show()

save_path = OUT_DIR / f'group_pfb_injection_pair{PAIR_ID}.png'
save_image_grid([img for _, img in items], save_path, nrow=cols)
print('saved:', save_path)


## Five Pairs: CSD Group Phi(S) Versus Phi(S_clean)

This repeats the `[0, 1, 2, 9]` group on all matched pairs. The output is small enough to inspect manually before running the metric cells.


In [ ]:
CSD_STYLE_GROUP = [0, 1, 2, 9]
PFB_ALPHA = 1.0
PFB_RANK = None
PFB_STRENGTH = 1.0
CLEAN_RANK = 8
CLEAN_STRENGTH = 1.0
CENTER_FEATURES = False

all_items = []
cols = 4
rows = len(matched_pairs)
plt.figure(figsize=(4 * cols, 4 * rows))

for pair_id, (pair_name, content_path, style_path) in enumerate(matched_pairs):
    content_x, _ = load_var_image(content_path)
    style_x, _ = load_var_image(style_path)

    with torch.no_grad():
        content_pack = image_to_features(content_x)
        style_pack = image_to_features(style_x)
        normal_img = pfb_inject_residual_scales(
            content_pack['deltas'],
            style_pack['deltas'],
            scale_ids=CSD_STYLE_GROUP,
            alpha=PFB_ALPHA,
            rank=PFB_RANK,
            strength=PFB_STRENGTH,
            center=CENTER_FEATURES,
            clean_style=False,
        )
        clean_img = pfb_inject_residual_scales(
            content_pack['deltas'],
            style_pack['deltas'],
            scale_ids=CSD_STYLE_GROUP,
            alpha=PFB_ALPHA,
            rank=PFB_RANK,
            strength=PFB_STRENGTH,
            center=CENTER_FEATURES,
            clean_style=True,
            clean_rank=CLEAN_RANK,
            clean_strength=CLEAN_STRENGTH,
        )

    row_items = [content_pack['recon'], style_pack['recon'], normal_img, clean_img]
    row_titles = ['content', 'style', 'group Phi(S)', 'group Phi(S_clean)']
    all_items.extend(row_items)

    for col, (title, img) in enumerate(zip(row_titles, row_items)):
        plt.subplot(rows, cols, pair_id * cols + col + 1)
        show_tensor_img(img, title if pair_id == 0 else None)
        if col == 0:
            plt.text(-0.04, 0.5, pair_name, transform=plt.gca().transAxes, rotation=90, va='center', ha='right', fontsize=11)

plt.suptitle('CSD-style group [0,1,2,9]: Phi(S) vs Phi(S_clean)')
plt.tight_layout()
plt.show()

save_path = OUT_DIR / 'five_pairs_csd_group_phi_vs_clean.png'
save_image_grid(all_items, save_path, nrow=cols)
print('saved:', save_path)


## Five Matched Pairs: Single-Scale PFB Sweep

Run this to see whether the optimal PFB scale is stable across different content/style pairs.


In [ ]:
PFB_ALPHA = 1.0
PFB_RANK = None
PFB_STRENGTH = 1.0
CENTER_FEATURES = False
KEEP_LATER_CONTENT = True

for pair_id, (pair_name, content_path, style_path) in enumerate(matched_pairs):
    content_x, _ = load_var_image(content_path)
    style_x, _ = load_var_image(style_path)

    with torch.no_grad():
        content_pack = image_to_features(content_x)
        style_pack = image_to_features(style_x)
        pfb_outputs = [
            pfb_inject_at_scale(
                content_pack['fhat_scales'],
                style_pack['fhat_scales'],
                content_pack['deltas'],
                scale_id=scale_id,
                alpha=PFB_ALPHA,
                rank=PFB_RANK,
                strength=PFB_STRENGTH,
                center=CENTER_FEATURES,
                keep_later_content=KEEP_LATER_CONTENT,
            )
            for scale_id in range(len(patch_nums))
        ]

    items = [('content recon', content_pack['recon']), ('style recon', style_pack['recon'])]
    items += [(f'PFB scale {i} ({pn}x{pn})', img) for i, (pn, img) in enumerate(zip(patch_nums, pfb_outputs))]
    cols = 4
    rows = math.ceil(len(items) / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, (title, img) in enumerate(items):
        plt.subplot(rows, cols, i + 1)
        show_tensor_img(img, title)
    plt.suptitle(f'{pair_name} | single-scale PFB sweep')
    plt.tight_layout()
    plt.show()

    safe_name = pair_name.replace(' ', '_').replace('+', 'plus').replace('/', '_')
    save_path = OUT_DIR / f'five_pairs_pfb_sweep_{pair_id:02d}_{safe_name}.png'
    save_image_grid([img for _, img in items], save_path, nrow=cols)
    print('saved:', save_path)


## Parameter Sweep: Alpha, Rank, Strength

Use this after the visual scale sweep.

Important meanings:

```text
alpha small -> keeps more singular components -> stronger style but more leakage
alpha large -> keeps mainly the first principal component -> safer but weaker
rank small  -> only top-k components are allowed
strength    -> interpolation between original content feature and full PFB result
```


In [ ]:
PAIR_ID = 0
PIVOT_SCALES_TO_TEST = [2, 3, 4, 7, 8, 9]
ALPHAS = [0.25, 0.5, 1.0, 2.0]
RANKS = [1, 2, 4, 8, None]
STRENGTHS = [0.5, 1.0]
CENTER_FEATURES = False
KEEP_LATER_CONTENT = True

pair_name, content_path, style_path = matched_pairs[PAIR_ID]
content_x, _ = load_var_image(content_path)
style_x, _ = load_var_image(style_path)

with torch.no_grad():
    content_pack = image_to_features(content_x)
    style_pack = image_to_features(style_x)

for pivot_scale in PIVOT_SCALES_TO_TEST:
    preview_items = [('content recon', content_pack['recon']), ('style recon', style_pack['recon'])]
    labels = []

    with torch.no_grad():
        for alpha in ALPHAS:
            for rank in [1, 4, None]:
                img = pfb_inject_at_scale(
                    content_pack['fhat_scales'],
                    style_pack['fhat_scales'],
                    content_pack['deltas'],
                    scale_id=pivot_scale,
                    alpha=alpha,
                    rank=rank,
                    strength=1.0,
                    center=CENTER_FEATURES,
                    keep_later_content=KEEP_LATER_CONTENT,
                )
                preview_items.append((f'a={alpha}, r={rank}', img))

    cols = 4
    rows = math.ceil(len(preview_items) / cols)
    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, (title, img) in enumerate(preview_items):
        plt.subplot(rows, cols, i + 1)
        show_tensor_img(img, title)
    plt.suptitle(f'{pair_name} | PFB parameter preview at scale {pivot_scale}')
    plt.tight_layout()
    plt.show()

    save_path = OUT_DIR / f'parameter_preview_pair{PAIR_ID}_scale{pivot_scale}.png'
    save_image_grid([img for _, img in preview_items], save_path, nrow=cols)
    print('saved:', save_path)


## Metrics

The paper evaluates style fidelity and text/content fidelity. Our image-only version records:

```text
content_similarity: similarity to content VAE reconstruction
style_similarity:   Gram-style similarity to style VAE reconstruction
balanced_score:     normalized average of both
```

This will tell us which scale is the best PFB intervention point for VAR-d20.


In [ ]:
from torchvision.models import vgg19, VGG19_Weights, resnet50, ResNet50_Weights

metric_device = device
imagenet_mean = torch.tensor([0.485, 0.456, 0.406], device=metric_device).view(1, 3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225], device=metric_device).view(1, 3, 1, 1)


def prep_metric_image(x, size=224):
    x = x.detach().float().to(metric_device)
    x = x.clamp(-1, 1).add(1).div(2)
    if x.shape[-2:] != (size, size):
        x = F.interpolate(x, size=(size, size), mode='bicubic', align_corners=False)
    return (x - imagenet_mean) / imagenet_std


try:
    content_metric_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(metric_device).eval()
    content_metric_name = 'dinov2_vits14'
except Exception as exc:
    print('DINOv2 load failed, falling back to ResNet50:', repr(exc))
    content_metric_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(metric_device).eval()
    content_metric_model.fc = torch.nn.Identity()
    content_metric_name = 'resnet50'

style_metric_model = vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features[:21].to(metric_device).eval()
for model in [content_metric_model, style_metric_model]:
    for p in model.parameters():
        p.requires_grad_(False)


@torch.no_grad()
def content_embedding(x):
    return F.normalize(content_metric_model(prep_metric_image(x)), dim=-1)


@torch.no_grad()
def gram_embedding(x):
    feats = style_metric_model(prep_metric_image(x))
    B, C, H, W = feats.shape
    flat = feats.view(B, C, H * W)
    gram = torch.bmm(flat, flat.transpose(1, 2)) / (C * H * W)
    return F.normalize(gram.flatten(1), dim=-1)


def cosine_to_float(a, b):
    return F.cosine_similarity(a, b, dim=-1).detach().cpu().item()

print('content metric:', content_metric_name)


In [ ]:
METRIC_DIR = OUT_DIR / 'metrics'
METRIC_DIR.mkdir(parents=True, exist_ok=True)

PFB_ALPHA = 1.0
PFB_RANK = None
PFB_STRENGTH = 1.0
CLEAN_RANK = 8
CLEAN_STRENGTH = 1.0
CENTER_FEATURES = False
KEEP_LATER_CONTENT = True
CSD_STYLE_GROUP = [0, 1, 2, 9]

records = []

for pair_id, (pair_name, content_path, style_path) in enumerate(tqdm(matched_pairs, desc='pairs')):
    content_x, _ = load_var_image(content_path)
    style_x, _ = load_var_image(style_path)

    with torch.no_grad():
        content_pack = image_to_features(content_x)
        style_pack = image_to_features(style_x)
        content_ref = content_embedding(content_pack['recon'])
        style_ref = gram_embedding(style_pack['recon'])

        for scale_id, pn in enumerate(tqdm(patch_nums, desc=pair_name, leave=False)):
            method_images = {
                'single_pfb_phi': pfb_inject_at_scale(
                    content_pack['fhat_scales'],
                    style_pack['fhat_scales'],
                    content_pack['deltas'],
                    scale_id=scale_id,
                    alpha=PFB_ALPHA,
                    rank=PFB_RANK,
                    strength=PFB_STRENGTH,
                    center=CENTER_FEATURES,
                    keep_later_content=KEEP_LATER_CONTENT,
                    clean_style=False,
                ),
                'single_pfb_phi_clean': pfb_inject_at_scale(
                    content_pack['fhat_scales'],
                    style_pack['fhat_scales'],
                    content_pack['deltas'],
                    scale_id=scale_id,
                    alpha=PFB_ALPHA,
                    rank=PFB_RANK,
                    strength=PFB_STRENGTH,
                    center=CENTER_FEATURES,
                    keep_later_content=KEEP_LATER_CONTENT,
                    clean_style=True,
                    clean_rank=CLEAN_RANK,
                    clean_strength=CLEAN_STRENGTH,
                ),
                'full_replace': full_feature_replace(
                    content_pack['fhat_scales'],
                    style_pack['fhat_scales'],
                    content_pack['deltas'],
                    scale_id=scale_id,
                    keep_later_content=KEEP_LATER_CONTENT,
                ),
            }

            for method_name, mixed_img in method_images.items():
                records.append({
                    'pair_id': pair_id,
                    'pair_name': pair_name,
                    'content_path': str(content_path.relative_to(WORKSPACE)),
                    'style_path': str(style_path.relative_to(WORKSPACE)),
                    'experiment_type': 'single_scale',
                    'method': method_name,
                    'inject_scale_id_0based': scale_id,
                    'inject_scale_id_1based': scale_id + 1,
                    'patch_size': pn,
                    'scale_group': str([scale_id]),
                    'alpha': PFB_ALPHA,
                    'rank': 'all_decay' if PFB_RANK is None else PFB_RANK,
                    'strength': PFB_STRENGTH,
                    'clean_rank': CLEAN_RANK if 'clean' in method_name else None,
                    'clean_strength': CLEAN_STRENGTH if 'clean' in method_name else None,
                    'keep_later_content': KEEP_LATER_CONTENT,
                    'content_similarity': cosine_to_float(content_embedding(mixed_img), content_ref),
                    'style_similarity': cosine_to_float(gram_embedding(mixed_img), style_ref),
                })

        group_method_images = {
            'group_0129_pfb_phi': pfb_inject_residual_scales(
                content_pack['deltas'],
                style_pack['deltas'],
                scale_ids=CSD_STYLE_GROUP,
                alpha=PFB_ALPHA,
                rank=PFB_RANK,
                strength=PFB_STRENGTH,
                center=CENTER_FEATURES,
                clean_style=False,
            ),
            'group_0129_pfb_phi_clean': pfb_inject_residual_scales(
                content_pack['deltas'],
                style_pack['deltas'],
                scale_ids=CSD_STYLE_GROUP,
                alpha=PFB_ALPHA,
                rank=PFB_RANK,
                strength=PFB_STRENGTH,
                center=CENTER_FEATURES,
                clean_style=True,
                clean_rank=CLEAN_RANK,
                clean_strength=CLEAN_STRENGTH,
            ),
        }

        for method_name, mixed_img in group_method_images.items():
            records.append({
                'pair_id': pair_id,
                'pair_name': pair_name,
                'content_path': str(content_path.relative_to(WORKSPACE)),
                'style_path': str(style_path.relative_to(WORKSPACE)),
                'experiment_type': 'group_scale',
                'method': method_name,
                'inject_scale_id_0based': None,
                'inject_scale_id_1based': None,
                'patch_size': None,
                'scale_group': str(CSD_STYLE_GROUP),
                'alpha': PFB_ALPHA,
                'rank': 'all_decay' if PFB_RANK is None else PFB_RANK,
                'strength': PFB_STRENGTH,
                'clean_rank': CLEAN_RANK if 'clean' in method_name else None,
                'clean_strength': CLEAN_STRENGTH if 'clean' in method_name else None,
                'keep_later_content': None,
                'content_similarity': cosine_to_float(content_embedding(mixed_img), content_ref),
                'style_similarity': cosine_to_float(gram_embedding(mixed_img), style_ref),
            })

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

stats_df = pd.DataFrame(records)
raw_csv = METRIC_DIR / 'pfb_phi_clean_single_and_group_raw.csv'
stats_df.to_csv(raw_csv, index=False)
print('saved raw stats:', raw_csv)
display(stats_df)


In [ ]:
single_stats_df = stats_df[stats_df['experiment_type'] == 'single_scale'].copy()
group_stats_df = stats_df[stats_df['experiment_type'] == 'group_scale'].copy()

single_summary_df = (
    single_stats_df
    .groupby(['method', 'inject_scale_id_1based', 'inject_scale_id_0based', 'patch_size'], as_index=False)
    .agg(
        content_mean=('content_similarity', 'mean'),
        content_std=('content_similarity', 'std'),
        style_mean=('style_similarity', 'mean'),
        style_std=('style_similarity', 'std'),
        n=('content_similarity', 'count'),
    )
)

group_summary_df = (
    group_stats_df
    .groupby(['method', 'scale_group'], as_index=False)
    .agg(
        content_mean=('content_similarity', 'mean'),
        content_std=('content_similarity', 'std'),
        style_mean=('style_similarity', 'mean'),
        style_std=('style_similarity', 'std'),
        n=('content_similarity', 'count'),
    )
)

# Balanced score is computed per experiment/method family.
def add_balanced_score(df, group_col='method'):
    df = df.copy()
    def minmax(series):
        denom = series.max() - series.min()
        if float(denom) == 0.0:
            return series * 0 + 1
        return (series - series.min()) / denom
    df['content_norm'] = df.groupby(group_col)['content_mean'].transform(minmax)
    df['style_norm'] = df.groupby(group_col)['style_mean'].transform(minmax)
    df['balanced_score'] = 0.5 * df['content_norm'] + 0.5 * df['style_norm']
    return df

single_summary_df = add_balanced_score(single_summary_df, group_col='method')
single_summary_df = single_summary_df.sort_values(['method', 'inject_scale_id_1based'])

group_summary_df['content_norm'] = group_summary_df['content_mean']
group_summary_df['style_norm'] = group_summary_df['style_mean']
group_summary_df['balanced_score'] = 0.5 * group_summary_df['content_mean'] + 0.5 * group_summary_df['style_mean']

everything_summary_csv = METRIC_DIR / 'pfb_phi_clean_single_and_group_summary.csv'
single_summary_csv = METRIC_DIR / 'pfb_phi_clean_single_scale_summary.csv'
group_summary_csv = METRIC_DIR / 'pfb_phi_clean_group_0129_summary.csv'

single_summary_df.to_csv(single_summary_csv, index=False)
group_summary_df.to_csv(group_summary_csv, index=False)
pd.concat([
    single_summary_df.assign(experiment_type='single_scale'),
    group_summary_df.assign(experiment_type='group_scale'),
], ignore_index=True, sort=False).to_csv(everything_summary_csv, index=False)

print('saved single summary:', single_summary_csv)
print('saved group summary:', group_summary_csv)
print('saved combined summary:', everything_summary_csv)

display(single_summary_df)
display(group_summary_df)

for method_name in single_summary_df['method'].unique():
    method_df = single_summary_df[single_summary_df['method'] == method_name]
    best_row = method_df.sort_values('balanced_score', ascending=False).iloc[0]
    print(
        f'Best single scale for {method_name}:',
        int(best_row.inject_scale_id_0based),
        f'({int(best_row.patch_size)}x{int(best_row.patch_size)})',
        '| content=', round(float(best_row.content_mean), 4),
        '| style=', round(float(best_row.style_mean), 4),
        '| balanced=', round(float(best_row.balanced_score), 4),
    )

print('\nCSD group [0,1,2,9] comparison:')
display(group_summary_df.sort_values('balanced_score', ascending=False))

plt.figure(figsize=(11, 5))
plot_methods = [
    ('single_pfb_phi', 'o'),
    ('single_pfb_phi_clean', '^'),
    ('full_replace', 's'),
]
for method_name, marker in plot_methods:
    method_df = single_summary_df[single_summary_df['method'] == method_name].sort_values('inject_scale_id_1based')
    x = method_df['inject_scale_id_1based']
    plt.plot(x, method_df['content_mean'], marker=marker, linestyle='-', label=f'{method_name} content')
    plt.plot(x, method_df['style_mean'], marker=marker, linestyle='--', label=f'{method_name} style')

scale_labels = single_summary_df[single_summary_df['method'] == 'single_pfb_phi'].sort_values('inject_scale_id_1based')
plt.xticks(
    scale_labels['inject_scale_id_1based'],
    [f"{int(s)}\n{int(p)}x{int(p)}" for s, p in zip(scale_labels['inject_scale_id_1based'], scale_labels['patch_size'])],
)
plt.xlabel('Injected scale')
plt.ylabel('Score')
plt.title(f'Phi(S) vs Phi(S_clean) vs full replacement | alpha={PFB_ALPHA}, rank={PFB_RANK}, strength={PFB_STRENGTH}')
plt.grid(alpha=0.25)
plt.legend(ncol=2)
plt.tight_layout()
plot_path = METRIC_DIR / 'pfb_phi_vs_clean_scale_curve.png'
plt.savefig(plot_path, dpi=160)
plt.show()
print('saved plot:', plot_path)


## What To Test And Conclude

Run the notebook in this order:

```text
1. Sanity check PFB vs full replacement at one pivot scale.
2. Run single-scale PFB sweep on one pair.
3. Run five-pair PFB sweep.
4. Run metrics for PFB vs full replacement.
5. Use the best balanced-score scale as the next candidate pivotal VAR scale.
```

Expected readings:

```text
If full replacement has high style but low content:
    the style reference feature contains content leakage.

If PFB improves content while keeping style:
    the paper's principal-feature decomposition transfers to VAR VAE space.

If one scale consistently wins:
    that scale is our VAR-d20 equivalent of the paper's pivotal stage.

If no scale works:
    the style signal likely needs transformer attention intervention, which becomes Notebook 4.
```
